### 256-SURPRISE LIBRARY DEMO (MATRIX FACTORIZATION)

In [ ]:
#Credits-Prof Eirinaki, Rashmi Sharma and Aditya Patel

### Make sure to install surprise library prior to running this code. Two options listed below.
#conda install -c conda-forge scikit-surprise
#!pip install scikit-surprise

#####if running on collab run this instead (and restart the environment as prompted)
#!pip install "numpy<2" scikit-surprise

### Importing libraries

In [1]:
from surprise import BaselineOnly
from surprise import Dataset
from surprise import Reader
from surprise.model_selection.split import train_test_split
from surprise.model_selection import cross_validate, GridSearchCV
import pandas as pd
import numpy as np
import os, io
from surprise import KNNBasic, KNNWithMeans
from surprise import SVDpp
from surprise import SVD
from surprise import accuracy

### Understanding the recommendations' generation problem
Basic recommender system design revolves around three fields user id,item id and ratings. 

The dataset consists of all three columns and they could be visualised as matrix containing userid as rows,item id as columns and ratings as data given by user for that item. As seen in class, major techniques to predict ratings of the user for an item are collaborative filtering and matrix factorization. We will work through the collaborative filtering  technique using Python's surprise library (https://surprise.readthedocs.io/en/stable/index.html) which provides a lot of built in function tailored to build recommender system. 

In [2]:
ratings_df  = pd.read_csv('datasets/movie_night_utilitymatrix_S26.csv')# read csv into ratings_df dataframe
ratings_df.head()

,user,movie,rating
0,1,2,5.0
1,1,3,5.0
2,1,7,3.0
3,1,11,2.0
4,1,14,5.0


In [3]:
reader = Reader(rating_scale=(1,5))  #invoke reader instance of surprise library
data=Dataset.load_from_df(ratings_df,reader) #load dataset into Surprise datastructure Dataset


### Training the model (Matrix factorization -based)

Before training the model, we need to create a training set. This needs to be distinct from any set used for cross-validation or testing/evaluation. 

There are several ways to perform hyperparameter tuning and/or evaluation. 

Surprise library provides several cross-validation iterators that allow to do the split from user-item matrix as below. (Ref: https://surprise.readthedocs.io/en/stable/getting_started.html#use-cross-validation-iterators)

##### Option 1: Holdout set

For latent-factor CF, surprise provides several inbuilt algorithms (SVD, SVDpp, NMF), which are implementations of the most popular matrix factorization algorithms. 

You can check them all here https://surprise.readthedocs.io/en/stable/prediction_algorithms_package.html

In [ ]:
#create training set
trainingSet, testSet = train_test_split(data, test_size=0.2, train_size=None, random_state=None, shuffle=True)

In [ ]:
#SVD Matrix Factorization
algo = SVD(n_factors=10, reg_all=0.01) #at the very least, set number of factors and regularization parameter
algo.fit(trainingSet)
predictions_svd = algo.test(testSet)

We will check the algorithm's accuracy or using the RMSE score of the predicted ratings.

In [ ]:
#validating rating predictions using RMSE
accuracy.rmse(predictions_svd, verbose=True) 

In [ ]:
# (optional, for demonstration) for each user-item combination in the test set we get predictions
predictions_svd

##### Option 2: Cross-validation

Run a cross validation procedure for a given algorithm, reporting accuracy measures and computation times.

You have several options in surprise library: https://surprise.readthedocs.io/en/stable/model_selection.html

In [ ]:
algo = SVD(n_factors=10, reg_all=0.01)
cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True) ##rerun the training part with different parameters

##### Option 3:  GridSearchCV
The GridSearchCV class computes accuracy metrics for an algorithm on various combinations of parameters, over a cross-validation procedure. This is useful for finding the best set of parameters for a prediction algorithm. It is analogous to GridSearchCV from scikit-learn.

In [ ]:
param_grid = {'n_factors': [5, 10, 20, 50, 100],
              'reg_all': [0.001, 0.01, 0.1],
              'n_epochs': [10, 20, 100]
              }


In [ ]:
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=5) 


In [ ]:
gs.fit(data)

In [ ]:
# best RMSE score
print(gs.best_score['rmse'])

In [ ]:
# combination of parameters that gave the best RMSE score
print(gs.best_params['rmse'])

In [ ]:
# We can now use the algorithm that yields the best rmse:
svd = gs.best_estimator['rmse']
svd.fit(data.build_full_trainset())

#You may use this instead of some parts of the following section, to make predictions for the unseen data (i.e. all the missing ratings)


### Example -- Making predictions for unknown ratings


#### UI prep
For our demo, we will create a user dictionary and movie dictionary where for user dictionary key is username and value userId which is used in our original dataset. For movie dictionary key is movieId and value is movie name. 

In [ ]:
user_df = pd.read_csv("user_name.csv")


In [ ]:
user_df.head(5)

In [ ]:
user_dict = {}
for i in range(len(user_df)):
    user_dict[user_df.iloc[i].username] = user_df.iloc[i].id

In [ ]:
movie_df = pd.read_csv("movie_name.csv")

In [ ]:
movie_df.head(5)

In [ ]:
movie_dict = {}
for i in range(len(movie_df)):
    movie_dict[movie_df.iloc[i].id] = movie_df.iloc[i].movieName

#### Find user-item pairs with no ratings

The build_anti_testset() function returns all the ratings that are not in the trainset, i.e. all the ratings 𝑟_{𝑢𝑖} where the user 𝑢 is known, the item 𝑖 is known, but the rating 𝑟_{𝑢𝑖} is not in the trainset. As 𝑟_{𝑢𝑖} is unknown, it is either replaced by the fill value or assumed to be equal to the mean of all ratings global_mean.

In [ ]:
# Retrieve the trainset.
trainset = data.build_full_trainset()


### Using previously trained model
svd = gs.best_estimator['rmse']
svd.fit(trainset)


### Alternative: Stand-alone version 
#for simplicity, we use the entire dataset, with the default algorithm to train the model. 
#In reality, you should follow one of the techniques above to find the optimal parameters.

### Build an algorithm, and train it. Follow methodology provided previously to perform hyperparameter tuning
###algo = SVD()    
###algo.fit(trainset)



# Find missing values and predict. 
anti_test_set = trainset.build_anti_testset() 
###predictions = algo.test(anti_test_set)
predictions = svd.test(anti_test_set)

The getMovieRecommendations function takes topN parameter which is how many movies you want to recommend to the users. It uses the predictions which we generated for the anti-test-set (i.e. the missing values).

In [ ]:
from collections import defaultdict

def getMovieRecommendations(topN):
    top_recs = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions: 
        top_recs[uid].append((iid, est))
     
    for uid, user_ratings in top_recs.items():
        user_ratings.sort(key = lambda x: x[1], reverse = True)
        top_recs[uid] = user_ratings[:topN]
     
    return top_recs 

In [ ]:
recommendations = getMovieRecommendations(3)

Fetch the movie name from movie dict and clean it

In [ ]:
def getMovieName(movie_id):
    if movie_id not in movie_dict:
        return ""
    m = movie_dict[movie_id].split('[')
    temp = m[1].split(']')
    return temp[0]

The getMovieRecommendationsForUser fuction takes username, and recommendations which we get from getMovieRecommendations function. 

In [ ]:
def getMovieRecommendationsForUser(userId, recommendations):
    if userId not in user_dict:
        print("User id is not present")
        return
    u_id = user_dict[userId]
    recommended_movies = recommendations[u_id]
    movie_list = []
    for movie in recommended_movies:
        movie_list.append((getMovieName(movie[0]),movie[1]))
    return movie_list    

In [ ]:
getMovieRecommendationsForUser('FALL-24-525',recommendations)

### Tips

1.Surprise dataset function just takes three columns,user-item and ratings so be careful.

2.Building Antitest set gives you all the unknown user-item ratings,you may not require all of them.

3.Explore more and have fun!
